# Séance 5 · Exercices — Git et GitHub · ⭐

**Niveau : ⭐ Débutant**

**Niveau de la séance : ⭐ Débutant** · chaque exercice porte son propre niveau (⭐ Débutant · ⭐⭐ Intermédiaire · ⭐⭐⭐ Avancé).

Comment travailler :
1. Lis l'énoncé, puis code dans la cellule « À toi » (les variables à remplir sont déjà nommées).
2. Lance la cellule de vérification : ✅ c'est bon, ❌ on réessaie (l'indice est là pour ça).
3. Ouvre la solution **seulement après avoir essayé**.

Ce notebook tourne dans **Google Colab** : rien à installer. Pour exécuter une cellule : clique dedans puis `Maj + Entrée`.

Les commandes de la leçon, dans un autre dépôt : celui du club cuisine du lycée, qui range ses recettes avec Git. Chaque exercice crée son propre dépôt neuf : tu peux les faire dans l'ordre que tu veux. Git répond en anglais : *untracked* = non suivi, *modified* = modifié, *deleted* = supprimé, *Changes to be committed* = prêt à être sauvegardé.

## Préparation

Une seule cellule à lancer. Elle crée un dossier d'entraînement et quelques outils :

- `nouveau_depot(nom, historique)` crée un dépôt neuf, déjà signé, avec éventuellement des commits faits d'avance ;
- `git("commande")` lance une commande Git dans le dernier dépôt créé et affiche la réponse (`git("status")` = `git status` dans un terminal) ;
- `ecrire(fichier, texte)` écrit un fichier, `lire(fichier)` le relit ;
- `nouveau_github(nom)` crée un faux GitHub (un dépôt nu) et `cloner(github, nom)` le copie sur un « autre ordinateur ».

In [ ]:
import os
import shlex
import shutil
import subprocess
import tempfile

ATELIER = tempfile.mkdtemp(prefix="exercices-git-")
COURANT = ATELIER

def _lancer(commande, dossier):
    r = subprocess.run(["git"] + shlex.split(commande), cwd=dossier, capture_output=True, text=True)
    return (r.stdout + r.stderr).strip()

def git(commande, dossier=None):
    """Lance « git <commande> » dans le dépôt courant (ou dans `dossier`) et affiche la réponse."""
    print(_lancer(commande, dossier or COURANT) or "(Git n'a rien répondu : tout s'est bien passé)")

def sortie(commande, dossier=None):
    """La réponse de Git, en texte, sans l'afficher (pour les vérifications)."""
    return _lancer(commande, dossier or COURANT)

def ecrire(fichier, texte, dossier=None):
    with open(os.path.join(dossier or COURANT, fichier), "w", encoding="utf-8") as f:
        f.write(texte)

def lire(fichier, dossier=None):
    chemin = os.path.join(dossier or COURANT, fichier)
    return open(chemin, encoding="utf-8").read() if os.path.exists(chemin) else None

def signer(dossier):
    _lancer('config user.name "Club cuisine"', dossier)
    _lancer('config user.email "club@lycee.local"', dossier)

def nouveau_depot(nom, historique=(), puis=None, supprimer=()):
    """Un dépôt neuf et signé. historique = (fichier, contenu, message) déjà commités, dans l'ordre ;
    puis = {fichier: contenu} écrits ensuite sans être sauvegardés ; supprimer = fichiers effacés ensuite."""
    global COURANT
    dossier = os.path.join(ATELIER, nom)
    shutil.rmtree(dossier, ignore_errors=True)
    os.makedirs(dossier)
    _lancer("init -q -b main", dossier)
    signer(dossier)
    for fichier, contenu, message in historique:
        ecrire(fichier, contenu, dossier)
        _lancer(f"add {fichier}", dossier)
        _lancer(f'commit -q -m "{message}"', dossier)
    for fichier, contenu in (puis or {}).items():
        ecrire(fichier, contenu, dossier)
    for fichier in supprimer:
        os.remove(os.path.join(dossier, fichier))
    COURANT = dossier
    return dossier

def nouveau_github(nom):
    dossier = os.path.join(ATELIER, nom + ".git")
    shutil.rmtree(dossier, ignore_errors=True)
    _lancer(f"init -q --bare -b main {dossier}", ATELIER)
    return dossier

def deux_ordinateurs(nom, historique):
    """Un faux GitHub qui contient `historique`, cloné sur deux ordinateurs : renvoie (github, cdi, maison)."""
    global COURANT
    github = nouveau_github(nom)
    depart = nouveau_depot(nom + "-depart", historique)
    _lancer(f"remote add origin {github}", depart)
    _lancer("push -q -u origin main", depart)
    COURANT = ATELIER
    return github, cloner(github, nom + "-cdi"), cloner(github, nom + "-maison")

def cloner(github, nom):
    dossier = os.path.join(ATELIER, nom)
    shutil.rmtree(dossier, ignore_errors=True)
    _lancer(f"clone -q {github} {dossier}", ATELIER)
    signer(dossier)
    return dossier

resultats = {}

def verifier(nom, condition):
    """Affiche ✅ ou ❌ sans jamais lever d'exception (condition = un booléen, ou une fonction sans argument)."""
    try:
        ok = bool(condition() if callable(condition) else condition)
    except Exception:
        ok = False
    resultats[nom] = ok
    print(("✅ " if ok else "❌ ") + nom + ("" if ok else " — pas encore, relis l'énoncé ou ouvre l'indice."))

print("Prêt ! Dossier d'entraînement :", ATELIER)

## Exercice 1 ⭐ · Le premier commit

Le club cuisine ouvre son dépôt. La cellule crée le dépôt et écrit la recette des crêpes dans `crepes.md`.
À toi de la **sauvegarder** : `add`, puis `commit` avec un message qui dit ce que tu ajoutes. Puis regarde l'historique avec `log --oneline`.

Résultat attendu : un commit, qui contient `crepes.md`.

<details><summary>Indice</summary>

Deux commandes : `git("add crepes.md")`, puis `git('commit -m "Ajoute la recette des crêpes"')`. Les guillemets doubles entourent le message, les simples entourent toute la commande.

</details>

In [ ]:
# À toi
depot_1 = nouveau_depot("club-cuisine", puis={"crepes.md": "# Crêpes\n\n- 250 g de farine\n- 4 œufs\n- 50 cl de lait\n"})
# tes commandes git ici

In [ ]:
verifier("Exercice 1", len(sortie("log --oneline", depot_1).splitlines()) == 1 and "crepes.md" in sortie("ls-files", depot_1))

<details><summary>Solution</summary>

```python
depot_1 = nouveau_depot("club-cuisine", puis={"crepes.md": "# Crêpes\n\n- 250 g de farine\n- 4 œufs\n- 50 cl de lait\n"})
git("add crepes.md")
git('commit -m "Ajoute la recette des crêpes"')
git("log --oneline")
```

</details>

## Exercice 2 ⭐ · Lire `git status`

Ce dépôt a déjà deux recettes sauvegardées. Depuis, quelqu'un a changé un fichier et en a créé un nouveau, sans rien sauvegarder.
Lance `git status`, lis la réponse, et remplis :

- `non_suivi` : le nom du fichier que Git ne suit pas encore,
- `modifie` : le nom du fichier suivi qui a changé.

<details><summary>Indice</summary>

*Untracked files* = non suivis ; *modified:* = modifié. Les noms de fichiers sont juste après.

</details>

In [ ]:
# À toi
depot_2 = nouveau_depot("menu-semaine", [
    ("menu.md", "# Menu\n\n- Lundi : pâtes\n", "Ajoute le menu de la semaine"),
    ("courses.md", "# Courses\n\n- tomates\n", "Ajoute la liste de courses"),
], puis={"courses.md": "# Courses\n\n- tomates\n- basilic\n", "budget.md": "# Budget\n\n30 euros\n"})
git("status")

non_suivi = "?"
modifie = "?"

In [ ]:
verifier("Exercice 2", non_suivi == "budget.md" and modifie == "courses.md")

<details><summary>Solution</summary>

```python
depot_2 = nouveau_depot("menu-semaine", [
    ("menu.md", "# Menu\n\n- Lundi : pâtes\n", "Ajoute le menu de la semaine"),
    ("courses.md", "# Courses\n\n- tomates\n", "Ajoute la liste de courses"),
], puis={"courses.md": "# Courses\n\n- tomates\n- basilic\n", "budget.md": "# Budget\n\n30 euros\n"})
git("status")

non_suivi = "budget.md"
modifie = "courses.md"
```

Non suivi : Git ne l'a jamais sauvegardé. Modifié : il est déjà dans l'historique, et il a changé depuis le dernier commit.

</details>

## Exercice 3 ⭐ · Ne sauvegarder qu'un seul fichier

Deux recettes ont changé : les crêpes (une erreur de dosage, pas encore corrigée) et les gaufres (terminées).
Sauvegarde **seulement les gaufres**, dans un commit à elles. Les crêpes doivent rester modifiées, hors du commit.

<details><summary>Indice</summary>

`git add` choisit précisément : `git("add gaufres.md")`, puis le commit. Vérifie avec `git("status")` : `crepes.md` doit encore apparaître comme *modified*.

</details>

In [ ]:
# À toi
depot_3 = nouveau_depot("desserts", [
    ("crepes.md", "# Crêpes\n\n- 250 g de farine\n", "Ajoute les crêpes"),
    ("gaufres.md", "# Gaufres\n\n- 200 g de farine\n", "Ajoute les gaufres"),
], puis={"crepes.md": "# Crêpes\n\n- 2500 g de farine\n", "gaufres.md": "# Gaufres\n\n- 200 g de farine\n- 2 œufs\n- sucre vanillé\n"})
# tes commandes git ici

In [ ]:
verifier("Exercice 3", sortie("show --name-only --format= HEAD", depot_3).split() == ["gaufres.md"] and "crepes.md" in sortie("status --short", depot_3) and len(sortie("log --oneline", depot_3).splitlines()) == 3)

<details><summary>Solution</summary>

```python
depot_3 = nouveau_depot("desserts", [
    ("crepes.md", "# Crêpes\n\n- 250 g de farine\n", "Ajoute les crêpes"),
    ("gaufres.md", "# Gaufres\n\n- 200 g de farine\n", "Ajoute les gaufres"),
], puis={"crepes.md": "# Crêpes\n\n- 2500 g de farine\n", "gaufres.md": "# Gaufres\n\n- 200 g de farine\n- 2 œufs\n- sucre vanillé\n"})
git("add gaufres.md")
git('commit -m "Termine la recette des gaufres"')
git("status")
```

C'est tout l'intérêt de l'étape `add` : un commit raconte une seule chose. Les crêpes auront le leur, une fois le dosage corrigé.

</details>

## Exercice 4 ⭐⭐ · Le bon message

Un bon message de commit dit **ce que le commit fait et pourquoi**, pour quelqu'un qui relira l'historique dans six mois.
Pour chacun des cinq messages, mets `True` s'il est bon, `False` sinon, dans la liste `bons` (dans l'ordre).

1. `"modif"`
2. `"Corrige la quantité de farine des crêpes (250 g, pas 2500)"`
3. `"update"`
4. `"Ajoute la recette du gâteau au yaourt"`
5. `"trucs"`

<details><summary>Indice</summary>

Demande-toi : en lisant seulement ce message dans `git log`, sais-tu ce qui a changé ?

</details>

In [ ]:
# À toi
bons = [None, None, None, None, None]

In [ ]:
verifier("Exercice 4", bons == [False, True, False, True, False])

<details><summary>Solution</summary>

```python
bons = [False, True, False, True, False]
```

« modif », « update », « trucs » sont vrais pour **tous** les commits : ils n'apprennent rien. Le verbe au début (« Ajoute », « Corrige ») dit ce que fait le commit.

</details>

## Exercice 5 ⭐⭐ · Lire l'historique

Le dépôt du club a déjà cinq commits. Avec `git log --oneline`, trouve :

- `nb_commits` : le nombre de commits,
- `plus_ancien` : le message du **premier** commit du dépôt (attention : `log` affiche le plus récent en haut),
- `nb_chocolat` : combien de messages parlent de chocolat.

<details><summary>Indice</summary>

`git("log --oneline")` : une ligne par commit, le numéro puis le message. Le plus ancien est **en bas**.

</details>

In [ ]:
# À toi
depot_5 = nouveau_depot("gateaux", [
    ("yaourt.md", "# Gâteau au yaourt\n", "Ajoute le gâteau au yaourt"),
    ("moelleux.md", "# Moelleux au chocolat\n", "Ajoute le moelleux au chocolat"),
    ("yaourt.md", "# Gâteau au yaourt\n\n- 1 pot de yaourt\n", "Précise les quantités du gâteau au yaourt"),
    ("cookies.md", "# Cookies aux pépites de chocolat\n", "Ajoute les cookies au chocolat"),
    ("moelleux.md", "# Moelleux au chocolat\n\n- 20 min à 180 °C\n", "Corrige la cuisson du moelleux"),
])
git("log --oneline")

nb_commits = None
plus_ancien = None
nb_chocolat = None

In [ ]:
verifier("Exercice 5", nb_commits == 5 and plus_ancien == "Ajoute le gâteau au yaourt" and nb_chocolat == 2)

<details><summary>Solution</summary>

```python
depot_5 = nouveau_depot("gateaux", [
    ("yaourt.md", "# Gâteau au yaourt\n", "Ajoute le gâteau au yaourt"),
    ("moelleux.md", "# Moelleux au chocolat\n", "Ajoute le moelleux au chocolat"),
    ("yaourt.md", "# Gâteau au yaourt\n\n- 1 pot de yaourt\n", "Précise les quantités du gâteau au yaourt"),
    ("cookies.md", "# Cookies aux pépites de chocolat\n", "Ajoute les cookies au chocolat"),
    ("moelleux.md", "# Moelleux au chocolat\n\n- 20 min à 180 °C\n", "Corrige la cuisson du moelleux"),
])
git("log --oneline")

nb_commits = 5
plus_ancien = "Ajoute le gâteau au yaourt"
nb_chocolat = 2
```

« Corrige la cuisson du moelleux » touche un gâteau au chocolat, mais le message ne dit pas « chocolat » : on compte les messages, pas les recettes. C'est aussi pour ça qu'un message précis aide à retrouver un commit.

</details>

## Exercice 6 ⭐⭐ · Ce qui a changé

Quelqu'un a modifié la recette de la pâte à pizza sans sauvegarder. Lance `git diff` et remplis :

- `ligne_retiree` : le texte de la ligne enlevée (sans le `-` du début),
- `ligne_ajoutee` : le texte de la ligne ajoutée à sa place (sans le `+`).

<details><summary>Indice</summary>

Dans la réponse de `git diff`, ignore les lignes qui commencent par `---` et `+++` (les noms de fichiers) : la ligne retirée commence par un seul `-`.

</details>

In [ ]:
# À toi
depot_6 = nouveau_depot("pizza", [
    ("pate.md", "# Pâte à pizza\n\n- 500 g de farine\n- 1 sachet de levure\n- 30 cl d'eau tiède\n", "Ajoute la pâte à pizza"),
], puis={"pate.md": "# Pâte à pizza\n\n- 500 g de farine\n- 1 sachet de levure\n- 32 cl d'eau tiède\n"})
git("diff")

ligne_retiree = None
ligne_ajoutee = None

In [ ]:
verifier("Exercice 6", ligne_retiree == "- 30 cl d'eau tiède" and ligne_ajoutee == "- 32 cl d'eau tiède")

<details><summary>Solution</summary>

```python
depot_6 = nouveau_depot("pizza", [
    ("pate.md", "# Pâte à pizza\n\n- 500 g de farine\n- 1 sachet de levure\n- 30 cl d'eau tiède\n", "Ajoute la pâte à pizza"),
], puis={"pate.md": "# Pâte à pizza\n\n- 500 g de farine\n- 1 sachet de levure\n- 32 cl d'eau tiède\n"})
git("diff")

ligne_retiree = "- 30 cl d'eau tiède"
ligne_ajoutee = "- 32 cl d'eau tiède"
```

Git ne compare pas des mots mais des lignes : changer un seul chiffre, c'est une ligne retirée et une ligne ajoutée. Ici, la ligne elle-même commence par un tiret de liste : on enlève seulement le signe que Git ajoute devant.

</details>

## Exercice 7 ⭐⭐ · Annuler une modification ratée

En voulant corriger la recette du guacamole, quelqu'un a tout effacé sauf le titre. Rien n'a été sauvegardé depuis.
Remets le fichier **exactement** comme au dernier commit, avec une seule commande.

<details><summary>Indice</summary>

La commande qui « restaure » un fichier depuis le dernier commit : `git("restore guacamole.md")`.

</details>

In [ ]:
# À toi
depot_7 = nouveau_depot("sauces", [
    ("guacamole.md", "# Guacamole\n\n- 3 avocats\n- 1 citron vert\n- 1 oignon rouge\n", "Ajoute le guacamole"),
], puis={"guacamole.md": "# Guacamole\n"})
print(lire("guacamole.md"))
# ta commande git ici

In [ ]:
verifier("Exercice 7", lire("guacamole.md", depot_7) == "# Guacamole\n\n- 3 avocats\n- 1 citron vert\n- 1 oignon rouge\n" and sortie("status --short", depot_7) == "")

<details><summary>Solution</summary>

```python
depot_7 = nouveau_depot("sauces", [
    ("guacamole.md", "# Guacamole\n\n- 3 avocats\n- 1 citron vert\n- 1 oignon rouge\n", "Ajoute le guacamole"),
], puis={"guacamole.md": "# Guacamole\n"})
print(lire("guacamole.md"))
git("restore guacamole.md")
print(lire("guacamole.md"))
```

Attention : `restore` jette les modifications pas encore sauvegardées. C'est ce qu'on veut ici ; sur un vrai travail, on vérifie d'abord avec `git diff` ce qu'on va perdre.

</details>

## Exercice 8 ⭐⭐ · Le fichier effacé par erreur

Le fichier `soupes.md` a été supprimé du dossier par erreur. Vérifie avec `git status` que Git l'a remarqué, puis fais-le revenir.

<details><summary>Indice</summary>

Même commande qu'à l'exercice 7 : Git garde le fichier dans le dernier commit, même s'il a disparu du dossier.

</details>

In [ ]:
# À toi
depot_8 = nouveau_depot("soupes", [
    ("soupes.md", "# Soupes\n\n- potiron\n- poireaux pommes de terre\n", "Ajoute les soupes d'hiver"),
], supprimer=["soupes.md"])
print(os.listdir(depot_8))
# tes commandes git ici

In [ ]:
verifier("Exercice 8", lire("soupes.md", depot_8) is not None and "potiron" in lire("soupes.md", depot_8) and sortie("status --short", depot_8) == "")

<details><summary>Solution</summary>

```python
depot_8 = nouveau_depot("soupes", [
    ("soupes.md", "# Soupes\n\n- potiron\n- poireaux pommes de terre\n", "Ajoute les soupes d'hiver"),
], supprimer=["soupes.md"])
print(os.listdir(depot_8))
git("status")
git("restore soupes.md")
print(lire("soupes.md"))
```

</details>

## Exercice 9 ⭐⭐⭐ · Revenir à une ancienne version, sans effacer l'histoire

La recette du chocolat chaud a été « améliorée » deux fois, et le club préférait la première version. On veut revenir à la version
du **premier** commit… tout en gardant les deux essais dans l'historique (on ne réécrit jamais l'histoire, on ajoute un commit).

1. `git restore --source=HEAD~2 chocolat.md` remet le fichier tel qu'il était deux commits avant le dernier ;
2. puis `add` et `commit`, avec un message qui dit qu'on revient à la recette d'origine.

Résultat attendu : le fichier est celui du premier commit, et l'historique compte **4** commits.

<details><summary>Indice</summary>

`HEAD` est le dernier commit, `HEAD~1` celui d'avant, `HEAD~2` encore avant : ici, le premier. Vérifie avec `git("show HEAD~2:chocolat.md")` avant de restaurer.

</details>

In [ ]:
# À toi
depot_9 = nouveau_depot("boissons", [
    ("chocolat.md", "# Chocolat chaud\n\n- 1 L de lait\n- 100 g de chocolat noir\n", "Ajoute le chocolat chaud"),
    ("chocolat.md", "# Chocolat chaud\n\n- 1 L de lait\n- 100 g de chocolat noir\n- 1 piment\n", "Essaie le chocolat au piment"),
    ("chocolat.md", "# Chocolat chaud\n\n- 1 L de lait\n- 100 g de chocolat noir\n- 1 piment\n- 1 pincée de sel\n", "Ajoute une pincée de sel"),
])
git("log --oneline")
# tes commandes git ici

In [ ]:
verifier("Exercice 9", lire("chocolat.md", depot_9) == "# Chocolat chaud\n\n- 1 L de lait\n- 100 g de chocolat noir\n" and len(sortie("log --oneline", depot_9).splitlines()) == 4 and sortie("status --short", depot_9) == "")

<details><summary>Solution</summary>

```python
depot_9 = nouveau_depot("boissons", [
    ("chocolat.md", "# Chocolat chaud\n\n- 1 L de lait\n- 100 g de chocolat noir\n", "Ajoute le chocolat chaud"),
    ("chocolat.md", "# Chocolat chaud\n\n- 1 L de lait\n- 100 g de chocolat noir\n- 1 piment\n", "Essaie le chocolat au piment"),
    ("chocolat.md", "# Chocolat chaud\n\n- 1 L de lait\n- 100 g de chocolat noir\n- 1 piment\n- 1 pincée de sel\n", "Ajoute une pincée de sel"),
])
git("log --oneline")
git("restore --source=HEAD~2 chocolat.md")
git("add chocolat.md")
git('commit -m "Revient à la recette d\'origine du chocolat chaud"')
git("log --oneline")
```

Les deux essais restent dans l'historique : si quelqu'un veut un jour retenter le piment, il le retrouvera. C'est la différence entre revenir en arrière et effacer.

</details>

## Exercice 10 ⭐⭐⭐ · Ce qui ne doit jamais partir

Le club a une clé pour l'appli de commande (`cle_commande.txt`) et un gros export des ventes (`ventes.csv`). Ni l'un ni l'autre
ne doit être sauvegardé. Écris un `.gitignore` qui les exclut, puis sauvegarde le `.gitignore` et la recette `tarte.md`.

Résultat attendu : `git status` n'affiche plus rien, et ni la clé ni le CSV ne sont dans le dépôt.

<details><summary>Indice</summary>

Le `.gitignore` est un simple fichier texte, un nom (ou un motif comme `*.csv`) par ligne : `ecrire(".gitignore", "cle_commande.txt\n*.csv\n")`. Puis `add` des deux fichiers et un commit.

</details>

In [ ]:
# À toi
depot_10 = nouveau_depot("commandes", puis={
    "tarte.md": "# Tarte aux pommes\n\n- 1 pâte brisée\n- 4 pommes\n",
    "cle_commande.txt": "cle-secrete-123",
    "ventes.csv": "jour,total\nlundi,42\n",
})
git("status --short")
# tes commandes ici

In [ ]:
verifier("Exercice 10", sortie("status --short", depot_10) == "" and sorted(sortie("ls-files", depot_10).split()) == [".gitignore", "tarte.md"])

<details><summary>Solution</summary>

```python
depot_10 = nouveau_depot("commandes", puis={
    "tarte.md": "# Tarte aux pommes\n\n- 1 pâte brisée\n- 4 pommes\n",
    "cle_commande.txt": "cle-secrete-123",
    "ventes.csv": "jour,total\nlundi,42\n",
})
git("status --short")
ecrire(".gitignore", "cle_commande.txt\n*.csv\n")
git("add .gitignore tarte.md")
git('commit -m "Ajoute la tarte aux pommes et ignore la clé et les exports"')
git("status --short")
```

Le `*.csv` protège aussi les prochains exports, quel que soit leur nom. Et si une clé part quand même sur GitHub, on la considère comme volée : on en demande une nouvelle.

</details>

## Exercice 11 ⭐⭐⭐ · Deux ordinateurs, un seul historique

Le club travaille au CDI et à la maison. La cellule crée un faux GitHub qui contient déjà la recette du crumble, et le clone sur les deux ordinateurs (`cdi` et `maison`).

1. Au **CDI** : ajoute une ligne à `crumble.md`, commit, puis `push`.
2. À la **maison** : `pull`, pour récupérer la ligne.

Toutes les commandes prennent `dossier=cdi` ou `dossier=maison`, pour dire sur quel ordinateur elles tournent.

<details><summary>Indice</summary>

Au CDI : `ecrire("crumble.md", ..., dossier=cdi)`, `git('commit -am "Ajoute le beurre"', dossier=cdi)` (le `-a` ajoute les fichiers suivis modifiés), `git("push", dossier=cdi)`. À la maison : `git("pull", dossier=maison)`.

</details>

In [ ]:
# À toi
github, cdi, maison = deux_ordinateurs("club", [("crumble.md", "# Crumble\n\n- 150 g de farine\n", "Ajoute le crumble")])
print(lire("crumble.md", maison))
# tes commandes ici

In [ ]:
verifier("Exercice 11", "100 g de beurre" in (lire("crumble.md", maison) or "") and sortie("log --oneline", maison) == sortie("log --oneline", cdi) and len(sortie("log --oneline", maison).splitlines()) == 2)

<details><summary>Solution</summary>

```python
github, cdi, maison = deux_ordinateurs("club", [("crumble.md", "# Crumble\n\n- 150 g de farine\n", "Ajoute le crumble")])
print(lire("crumble.md", maison))
ecrire("crumble.md", "# Crumble\n\n- 150 g de farine\n- 100 g de beurre\n", dossier=cdi)
git('commit -am "Ajoute le beurre au crumble"', dossier=cdi)
git("push", dossier=cdi)
git("pull", dossier=maison)
print(lire("crumble.md", maison))
```

Le réflexe des gens qui travaillent sur plusieurs machines : `pull` en arrivant, `push` en partant. Sinon, deux versions différentes finissent par se croiser.

</details>

## Exercice 12 ⭐⭐⭐ · Défi : le dépôt d'un projet, de zéro

Monte de zéro le dépôt d'un mini-projet et envoie-le sur un faux GitHub :

1. un `README.md` avec un titre (`# …`) et au moins deux sous-titres (`## …`) ;
2. un `.gitignore` qui exclut `donnees.csv` ;
3. au moins **deux** commits, chacun avec un vrai message (pas « modif », « update », « test ») ;
4. un `push` vers `github_projet`.

Le dossier et le faux GitHub sont créés par la cellule ; les données `donnees.csv` aussi, et elles ne doivent pas partir.

<details><summary>Indice</summary>

Un commit pour le `README.md`, un autre pour le `.gitignore` (ou une amélioration du README). Puis `git(f"remote add origin {github_projet}")` et `git("push -u origin main")`. Vérifie sur le faux GitHub avec `git("log --oneline", dossier=github_projet)`.

</details>

In [ ]:
# À toi
github_projet = nouveau_github("projet")
depot_12 = nouveau_depot("mon-projet", puis={"donnees.csv": "nom,score\nNova,12\n"})
# tes commandes ici

In [ ]:
verifier("Exercice 12", (lambda log, fichiers, readme: len(log) >= 2 and all(m.split(" ", 1)[1].strip().lower() not in ("modif", "update", "test", "maj") for m in log)
         and "README.md" in fichiers and ".gitignore" in fichiers and "donnees.csv" not in fichiers
         and readme.startswith("# ") and readme.count("\n## ") >= 2)(
         sortie("log --oneline main", github_projet).splitlines(), sortie("ls-tree -r --name-only main", github_projet).split(), sortie("show main:README.md", github_projet)))

<details><summary>Solution</summary>

```python
github_projet = nouveau_github("projet")
depot_12 = nouveau_depot("mon-projet", puis={"donnees.csv": "nom,score\nNova,12\n"})
ecrire("README.md", "# Les scores du club\n\n## Le projet\nComparer les scores des joueurs du club.\n\n## Les données\nUn fichier CSV, gardé hors du dépôt.\n")
git("add README.md")
git('commit -m "Ajoute le README du projet"')
ecrire(".gitignore", "donnees.csv\n")
git("add .gitignore")
git('commit -m "Garde les données hors du dépôt"')
git(f"remote add origin {github_projet}")
git("push -u origin main")
git("log --oneline", dossier=github_projet)
```

C'est exactement ce que tu as fait sur le vrai GitHub en séance, en version commandes : un README qui explique, des données gardées à l'abri, un historique qui raconte le projet.

</details>

## Bilan

La cellule ci-dessous compte tes ✅. Relance-la après chaque exercice réussi.

In [ ]:
reussis = sum(1 for ok in resultats.values() if ok)
print(f"{reussis} exercice(s) réussi(s) sur {len(resultats)}")
for nom, ok in resultats.items():
    print("  ", "✅" if ok else "❌", nom)